# v13 — model fondasi EKG ECGFounder 1-lead pada window SIHEDAF

Rancangan dikunci di [RANCANGAN.md](RANCANGAN.md) sebelum metrik dihitung (sha256 di `logs/v13_praregistrasi.log`).
ECGFounder (PKUDigitalHealth, MIT; 10,8 juta EKG, 1,8 juta pasien, 150 label termasuk sinus takikardia dan AFL) dipakai beku:
zero-shot dari head aslinya, probe regresi logistik pada fitur 1.024 dimensi, dan fusi dengan v3.1. Jalur **50hz** meniru
perangkat; jalur **250hz** adalah kontrol untuk mengukur kerugian sampling rendah.

Butuh `../data/ecgfounder/` (bobot + kode resmi, diunduh dengan `tools/unduh_berkas.py`, SHA256 dari API Hugging Face) dan GPU
untuk inferensi (±2.000 window/detik di RTX 5060). Dijalankan dengan cwd = folder ini.

In [1]:
import json, sys, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
sys.path[:0] = ["../v11_model_tim", "../v3.3_takikardia_asli"]
import ecgfounder as E
import takikardia_asli as K
from bersama import data as D, evaluasi as Ev
d = D.muat()
for jalur in ("50hz", "250hz"):
    E.jalankan_afdb(jalur)                         # dilewati per rekaman bila cache sudah ada
print("label head yang dipakai:", E.indeks_label())

label head yang dipakai: {'NSR': 1, 'SR': 3, 'AF': 5, 'ST': 6, 'AFL': 32, 'SVT': 93}


## 1. AFDB, protokol penuh (fold resmi 5×5, bias cross-fit)

In [2]:
oof = {}
for jalur in ("50hz", "250hz"):
    lo, fi = E.muat_afdb(jalur)
    oof[f"zero_{jalur}"] = np.repeat(E.skor_zero_shot(lo)[None], 5, 0)
    p = E.CACHE / f"oof_probe_{jalur}.npz"
    if not p.exists():
        np.savez_compressed(p, oof=E.oof_probe(fi, d.y, d.folds))
    oof[f"probe_{jalur}"] = np.load(p)["oof"]
    oof[f"fusi_{jalur}"] = E.fusi(oof[f"probe_{jalur}"], d.oof_v31)
p31, b31 = Ev.keputusan(d.oof_v31, d.folds, d.y)
pred, bias = {"v31": p31}, {"v31": b31}
for k, o in oof.items():
    pred[k], bias[k] = Ev.keputusan(o, d.folds, d.y)
tab = Ev.tabel_metrik(d, pred)
pd.DataFrame({k: {m: f"{v[0]:.3f} [{v[1]:.2f}–{v[2]:.2f}]" for m, v in tab[k].items() if m in ("Macro-F1","AFL F1","AFL R","AFL P","AFIB F1","Normal R")} for k in pred})

,v31,zero_50hz,probe_50hz,fusi_50hz,zero_250hz,probe_250hz,fusi_250hz
Macro-F1,0.832 [0.67–0.92],0.663 [0.64–0.68],0.746 [0.62–0.81],0.775 [0.66–0.82],0.649 [0.62–0.67],0.661 [0.63–0.69],0.750 [0.68–0.79]
Normal R,0.924 [0.84–0.97],0.973 [0.96–0.99],0.944 [0.91–0.97],0.973 [0.95–0.99],0.938 [0.88–0.98],0.957 [0.93–0.98],0.981 [0.96–0.99]
AFIB F1,0.909 [0.86–0.94],0.931 [0.85–0.97],0.915 [0.89–0.94],0.938 [0.92–0.96],0.932 [0.85–0.98],0.920 [0.89–0.95],0.951 [0.93–0.97]
AFL P,0.715 [0.26–0.95],0.053 [0.03–0.06],0.393 [0.01–0.59],0.394 [0.06–0.52],0.033 [0.02–0.05],0.117 [0.04–0.30],0.326 [0.09–0.60]
AFL R,0.592 [0.13–0.91],0.256 [0.03–0.36],0.366 [0.01–0.58],0.468 [0.08–0.69],0.253 [0.03–0.69],0.147 [0.05–0.45],0.349 [0.11–0.52]
AFL F1,0.647 [0.17–0.89],0.087 [0.03–0.10],0.379 [0.01–0.55],0.423 [0.07–0.56],0.058 [0.03–0.07],0.118 [0.05–0.17],0.332 [0.11–0.44]


In [3]:
bd = {k: Ev.banding(d, pred[k], p31) for k in oof}
b5025 = {m: Ev.banding(d, pred[f"{m}_50hz"], pred[f"{m}_250hz"]) for m in ("zero", "probe", "fusi")}
display(pd.DataFrame({k: {m: f"Δ {v[0]:+.3f} [{v[1]:+.3f}…{v[2]:+.3f}], P = {v[3]:.3f} → {Ev.vonis(v[3])}" for m, v in b.items()} for k, b in bd.items()}).T)
display(pd.DataFrame({k: {m: f"Δ {v[0]:+.3f} [{v[1]:+.3f}…{v[2]:+.3f}], P = {v[3]:.3f}" for m, v in b.items()} for k, b in b5025.items()}).T.rename_axis("50 Hz − 250 Hz"))

,Macro-F1,AFL F1,AFL P,AFL R,Normal R
zero_50hz,"Δ -0.168 [-0.258…+0.005], P = 0.026 → lebih buruk","Δ -0.560 [-0.850…-0.086], P = 0.019 → lebih buruk","Δ -0.663 [-0.890…-0.217], P = 0.004 → lebih buruk","Δ -0.336 [-0.878…+0.182], P = 0.063 → tidak te...","Δ +0.049 [+0.001…+0.129], P = 0.980 → lebih baik"
probe_50hz,"Δ -0.085 [-0.290…+0.020], P = 0.087 → tidak te...","Δ -0.269 [-0.858…+0.006], P = 0.038 → lebih buruk","Δ -0.322 [-0.877…-0.061], P = 0.003 → lebih buruk","Δ -0.226 [-0.877…+0.083], P = 0.247 → tidak te...","Δ +0.020 [-0.038…+0.106], P = 0.686 → tidak te..."
fusi_50hz,"Δ -0.057 [-0.231…+0.032], P = 0.187 → tidak te...","Δ -0.224 [-0.731…+0.014], P = 0.059 → tidak te...","Δ -0.321 [-0.663…-0.111], P = 0.003 → lebih buruk","Δ -0.124 [-0.764…+0.188], P = 0.315 → tidak te...","Δ +0.049 [+0.006…+0.126], P = 0.994 → lebih baik"
zero_250hz,"Δ -0.183 [-0.262…-0.012], P = 0.021 → lebih buruk","Δ -0.589 [-0.851…-0.120], P = 0.014 → lebih buruk","Δ -0.682 [-0.913…-0.234], P = 0.003 → lebih buruk","Δ -0.339 [-0.869…+0.527], P = 0.101 → tidak te...","Δ +0.014 [-0.064…+0.107], P = 0.639 → tidak te..."
probe_250hz,"Δ -0.171 [-0.268…-0.031], P = 0.007 → lebih buruk","Δ -0.529 [-0.797…-0.114], P = 0.002 → lebih buruk","Δ -0.598 [-0.856…-0.218], P = 0.000 → lebih buruk","Δ -0.444 [-0.784…+0.056], P = 0.095 → tidak te...","Δ +0.033 [-0.021…+0.117], P = 0.834 → tidak te..."
fusi_250hz,"Δ -0.081 [-0.197…+0.015], P = 0.052 → tidak te...","Δ -0.316 [-0.642…-0.053], P = 0.021 → lebih buruk","Δ -0.389 [-0.645…-0.077], P = 0.009 → lebih buruk","Δ -0.243 [-0.665…+0.080], P = 0.095 → tidak te...","Δ +0.057 [+0.014…+0.137], P = 1.000 → lebih baik"


,Macro-F1,AFL F1,AFL P,AFL R,Normal R
50 Hz − 250 Hz,,,,,
zero,"Δ +0.015 [-0.003…+0.026], P = 0.957","Δ +0.029 [-0.004…+0.043], P = 0.956","Δ +0.020 [-0.004…+0.032], P = 0.964","Δ +0.003 [-0.341…+0.073], P = 0.523","Δ +0.035 [+0.001…+0.079], P = 0.989"
probe,"Δ +0.086 [-0.029…+0.131], P = 0.740","Δ +0.260 [-0.077…+0.402], P = 0.716","Δ +0.276 [-0.130…+0.425], P = 0.802","Δ +0.218 [-0.294…+0.422], P = 0.661","Δ -0.013 [-0.048…+0.021], P = 0.217"
fusi,"Δ +0.025 [-0.047…+0.050], P = 0.684","Δ +0.092 [-0.134…+0.172], P = 0.708","Δ +0.068 [-0.245…+0.177], P = 0.549","Δ +0.119 [-0.177…+0.257], P = 0.672","Δ -0.008 [-0.030…+0.011], P = 0.203"


In [4]:
afl = {k: Ev.afl_per_pasien(d, pred[k]) for k in pred}
palsu = {k: Ev.afl_palsu(d, pred[k]) for k in pred}
display(pd.DataFrame(afl).round(2).rename_axis("recall AFL per pasien"))
display(pd.DataFrame(palsu).round(0).rename_axis("AFL palsu (window/ulangan)"))
bias_info = {k: dict(median=np.median(b.reshape(-1, 3), 0).tolist(), di_batas=Ev.bias_di_batas(b)) for k, b in bias.items()}
pd.DataFrame(bias_info).T

,v31,zero_50hz,probe_50hz,fusi_50hz,zero_250hz,probe_250hz,fusi_250hz
recall AFL per pasien,,,,,,,
04043,0.40,0.80,0.40,0.40,0.80,0.00,0.20
04908,0.92,0.28,0.31,0.57,0.37,0.99,1.00
04936,0.51,0.35,0.60,0.70,0.27,0.14,0.43
06426,0.17,0.13,0.13,0.39,0.13,0.32,0.44
07910,0.00,0.40,0.00,0.00,0.88,0.00,0.00
08215,0.00,0.00,0.11,0.11,0.00,0.00,0.00
08378,0.96,0.01,0.00,0.10,0.01,0.07,0.19


,v31,zero_50hz,probe_50hz,fusi_50hz,zero_250hz,probe_250hz,fusi_250hz
AFL palsu (window/ulangan),,,,,,,
total,274.0,5320.0,653.0,903.0,8600.0,2580.0,1112.0
dari_normal,55.0,826.0,339.0,337.0,3936.0,2211.0,814.0
dari_afib,219.0,4494.0,315.0,566.0,4664.0,368.0,299.0
pasien_tanpa_afl,25.0,367.0,179.0,109.0,1157.0,323.0,75.0


,median,di_batas
v31,"[0.0, -0.75, -4.5]",0.0
zero_50hz,"[0.0, 1.75, 3.0]",0.42
probe_50hz,"[0.0, -1.25, -4.5]",0.02
fusi_50hz,"[0.0, -1.5, -2.5]",0.0
zero_250hz,"[0.0, 1.75, 2.5]",0.32
probe_250hz,"[0.0, -4.5, -5.0]",0.08
fusi_250hz,"[0.0, -3.0, -3.25]",0.06


In [5]:
tanpa = Ev.tanpa_04936(d, {"v31": d.oof_v31, "probe_50hz": oof["probe_50hz"], "fusi_50hz": oof["fusi_50hz"]}, list(range(5)))
ep = {}
for k in ("v31", "probe_50hz", "fusi_50hz"):
    for kelas, n in ((1, "AFIB"), (2, "AFL")):
        ms = [K.metrik_episode(d.records, d.t, d.y, pred[k][r], kelas) for r in range(5)]
        ep[f"{k} {n}"] = {m: float(np.mean([x[m] for x in ms])) for m in ms[0]}
print("tanpa 04936:", json.dumps({k: [round(x, 3) for x in v["Macro-F1"]] for k, v in tanpa["rows"].items()}))
pd.DataFrame(ep).T.round(3)

tanpa 04936: {"v31": [0.859, 0.622, 0.923], "probe_50hz": [0.629, 0.609, 0.653], "fusi_50hz": [0.676, 0.628, 0.725]}


,se_episode,ppv_episode,se_durasi,ppv_durasi,n_ref,n_det,palsu_per_jam
v31 AFIB,0.970,0.562,0.932,0.887,881.0,6252.4,11.858
v31 AFL,0.686,0.403,0.592,0.715,14.0,198.2,0.521
probe_50hz AFIB,0.944,0.604,0.918,0.913,881.0,6740.2,11.545
probe_50hz AFL,0.529,0.169,0.366,0.393,14.0,431.0,1.553
fusi_50hz AFIB,0.934,0.719,0.922,0.956,881.0,4802.0,5.779
fusi_50hz AFL,0.714,0.148,0.468,0.394,14.0,396.8,1.473


## 2. H2: sinus takikardia asli (EKG latihan fisik v3.3, mode perangkat, jalur 50 Hz)

In [6]:
lo50, fi50 = E.muat_afdb("50hz")
probe, mu, sd = E.model_probe_penuh(fi50, d.y)
b_probe, b_fusi, b_zero = (E.bias_median(oof[k], d.y, d.folds) for k in ("probe_50hz", "fusi_50hz", "zero_50hz"))
def keputusan_eksternal(lo, fi, p_paket):
    pz = E.cv.apply_bias(E.skor_zero_shot(lo), b_zero).argmax(1)
    pp = probe.predict_proba((fi.astype(np.float32) - mu) / sd)
    kp = E.cv.apply_bias(pp, b_probe).argmax(1)
    kf = E.cv.apply_bias(E.fusi(pp, E.tanpa_bias_paket(p_paket)), b_fusi).argmax(1) if p_paket is not None else None
    return pz, kp, kf
J = K.muat_jendela(); okj = J["ok"]
w50, baris = E.jendela_latihan()
lo_l, fi_l = E.jalankan_eksternal("latihan_ok_50hz", w50)
pos = {i: j for j, i in enumerate(np.flatnonzero(okj))}               # baris jendela → kolom prediksi v3.3
p31_l = np.load(K.CACHE / "prediksi.npz")["p_v31"][[pos[i] for i in baris]]
kz, kp, kf = keputusan_eksternal(lo_l, fi_l, p31_l)
hr_l, subj_l, sumber_l = J["hr"][baris], J["subjek"][baris], J["sumber"][baris]
h2 = {}
for nama, kk in (("v3.1 paket", p31_l.argmax(1)), ("zero-shot", kz), ("probe", kp), ("fusi", kf)):
    m = hr_l >= 120
    t, c, _ = K._boot_subjek(subj_l[m], (kk[m] != 0).astype(float))
    h2[nama] = dict(alarm=t, ci=c.tolist(), n=int(m.sum()), per_sumber={s: float((kk[m & (sumber_l == s)] != 0).mean()) for s in np.unique(sumber_l[m])})
pd.DataFrame({k: {"alarm ≥120 bpm": f"{v['alarm']*100:.1f}% [{v['ci'][0]*100:.0f}–{v['ci'][1]*100:.0f}]", **{s: f"{x*100:.0f}%" for s, x in v["per_sumber"].items()}} for k, v in h2.items()}).T

,alarm ≥120 bpm,EPFL,EPHNOGRAM,Wrist
v3.1 paket,98.4% [97–100],100%,98%,100%
zero-shot,5.4% [3–9],5%,6%,0%
probe,30.7% [23–40],10%,33%,2%
fusi,62.0% [52–71],53%,64%,8%


## 3. H4: data eksternal berlabel — PTB-XL (lead I, 100 → 50 Hz) dan Icentia11k (window v10, akademik saja)

In [7]:
P = E.ptbxl()
w50p, _ = E.ke_50hz(P["ecg"], P["fs"])
lo_p, fi_p = E.jalankan_eksternal("ptbxl_50hz", w50p)
p31_p = E.rr_paket_v31(P["ecg"], P["fs"])
okp = np.isfinite(p31_p).all(1)
kz, kp, kf = keputusan_eksternal(lo_p[okp], fi_p[okp], p31_p[okp])
yp, pp_ = P["y"][okp], P["pasien"][okp]
ptb = {nama: E.banding_pasien(yp, kk, p31_p[okp].argmax(1), pp_) for nama, kk in (("zero-shot", kz), ("probe", kp), ("fusi", kf))}
st = P["st"][okp]
ptb_st = {nama: float((kk[st] != 0).mean()) for nama, kk in (("v3.1 paket", p31_p[okp].argmax(1)), ("zero-shot", kz), ("probe", kp), ("fusi", kf))}
display(pd.DataFrame({k: {"Macro-F1 (model / v3.1)": f"{v['titik_a'][0]:.3f} / {v['titik_b'][0]:.3f}", "Δ Macro-F1, P": f"{v['Macro-F1'][0]:+.3f}, P = {v['Macro-F1'][3]:.3f}",
                          "recall AFL (model / v3.1)": f"{v['titik_a'][1]:.2f} / {v['titik_b'][1]:.2f}"} for k, v in ptb.items()}).T)
print("PTB-XL: alarm AFIB/AFL pada sinus takikardia (STACH, n =", int(st.sum()), "):", {k: round(v, 3) for k, v in ptb_st.items()})

,Macro-F1 (model / v3.1),"Δ Macro-F1, P",recall AFL (model / v3.1)
zero-shot,0.642 / 0.622,"+0.020, P = 0.954",0.62 / 0.50
probe,0.559 / 0.622,"-0.063, P = 0.004",0.08 / 0.50
fusi,0.658 / 0.622,"+0.036, P = 0.950",0.24 / 0.50


PTB-XL: alarm AFIB/AFL pada sinus takikardia (STACH, n = 811 ): {'v3.1 paket': 0.297, 'zero-shot': 0.035, 'probe': 0.07, 'fusi': 0.059}


In [8]:
zi = np.load(E.REPO / "cache/v10/icentia_windows.npz")
lo_i, fi_i = E.jalankan_eksternal("icentia_50hz", zi["ecg"].astype(np.float32))
from predictor import ECGPredictor
P31 = ECGPredictor(str(E.REPO / "v3.1_kalibrasi_produksi/model_export"), name="sihedaf_v3_1", device="cpu")
p31_i = P31.predict_proba_sequences(zi["X"], zi["M"])
kz, kp, kf = keputusan_eksternal(lo_i, fi_i, p31_i)
ico = {nama: E.banding_pasien(zi["y"], kk, p31_i.argmax(1), zi["patient"]) for nama, kk in (("zero-shot", kz), ("probe", kp), ("fusi", kf))}
pd.DataFrame({k: {"Macro-F1 (model / v3.1)": f"{v['titik_a'][0]:.3f} / {v['titik_b'][0]:.3f}", "Δ Macro-F1, P": f"{v['Macro-F1'][0]:+.3f}, P = {v['Macro-F1'][3]:.3f}",
                  "recall AFL (model / v3.1)": f"{v['titik_a'][1]:.2f} / {v['titik_b'][1]:.2f}"} for k, v in ico.items()}).T

,Macro-F1 (model / v3.1),"Δ Macro-F1, P",recall AFL (model / v3.1)
zero-shot,0.549 / 0.476,"+0.073, P = 1.000",0.14 / 0.05
probe,0.445 / 0.476,"-0.031, P = 0.000",0.02 / 0.05
fusi,0.481 / 0.476,"+0.004, P = 0.970",0.05 / 0.05


In [9]:
ring = {"metrik": tab, "banding_v31": bd, "banding_50_250": b5025, "afl_per_pasien": afl, "afl_palsu": palsu, "bias": bias_info,
        "tanpa_04936": tanpa, "episode": ep, "h2_latihan": h2, "ptbxl": ptb, "ptbxl_st": ptb_st, "icentia": ico,
        "n": {"ptbxl": int(okp.sum()), "ptbxl_st": int(st.sum()), "icentia": int(len(zi["y"])), "latihan_≥120": int((hr_l >= 120).sum())},
        "lingkungan": Ev._lingkungan()}
(E.CACHE / "ringkasan.json").write_text(json.dumps(ring, indent=1, default=float, ensure_ascii=False))
print("tersimpan:", E.CACHE / "ringkasan.json")

tersimpan: /mnt/HDD/03_Akademik_dan_Tugas/WrapSihedaf/cache/v13/ringkasan.json


## 4. Kesimpulan

Dijalankan di PC (RTX 5060) lewat `tools/run_notebook.py`, 6 November 2026 (±20 menit dengan cache inferensi AFDB; inferensi AFDB dua jalur ±8 menit, probe 25 fold × 2 jalur ±35 menit saat CPU dibagi).

- **Sinus takikardia asli (H2) — perbaikan besar.** EKG latihan fisik, sinus ≥ 120 bpm (mode perangkat, 50 Hz): alarm AFIB/AFL zero-shot ECGFounder **5,4% [3–9]** vs paket v3.1 **98,4%**; probe 30,7%, fusi 62,0%. PTB-XL sinus takikardia (STACH, 811): v3.1 29,7% → zero-shot 3,5%.
- **Data eksternal berlabel (H4).** Zero-shot vs v3.1: PTB-XL Macro-F1 0,642 vs 0,622 (P = 0,954, lebih baik), recall AFL 0,62 vs 0,50; Icentia 0,549 vs 0,476 (P = 1,00), recall AFL 0,14 vs 0,05. Probe (dilatih di AFDB) lebih buruk di keduanya (P ≤ 0,004).
- **AFDB (H1) — lebih buruk untuk AFL.** Macro-F1: zero-shot 0,663 (P = 0,026, lebih buruk), probe 0,746 (P = 0,087), fusi 0,775 (P = 0,187) vs v3.1 0,832. AFL 2:1 pasien 08378 hilang (zero-shot 0,01, probe 0,00 vs 0,96); zero-shot menyebut banyak AFIB sebagai AFL (AFL palsu 5.320 window/ulangan, 4.494 dari AFIB). Bias zero-shot menyentuh batas grid di 42% fold (tanda bahaya v9). Tanpa 04936: probe 0,629, fusi 0,676 vs v3.1 0,859.
- **Biaya 50 Hz (H3):** tidak terlihat. Jalur 50 Hz justru tidak lebih buruk dari kontrol 250 Hz (probe +0,086, P = 0,74; zero-shot +0,015, P = 0,96).
- **Episode (gaya EC57):** fusi menurunkan episode AFIB palsu 11,9 → 5,8/jam, tetapi episode AFL palsu naik 0,52 → 1,47/jam.
- **Makna:** pengetahuan pralatih dari 1,8 juta pasien mengenali sinus takikardia dari bentuk gelombang (gelombang P) bahkan di 50 Hz, sesuatu yang tidak bisa dilakukan model RR. Harganya: AFL 2:1 di ±150 bpm, yang mirip sinus takikardia, ikut disebut sinus. Ini trade-off yang sama dengan v7/v8 tetapi jauh lebih kuat di sisi sinus takikardia dan lebih tahan antar-dataset.
- **Tidak bisa disimpulkan:** apakah AFL 2:1 di perangkat SIHEDAF akan hilang dengan cara yang sama (AFDB: satu pasien, 08378). Model 370 MB hanya cocok di backend.